# Textförbehandling och feature engineering

Den här notebooken förbereder den rensade Amazon All_Beauty-datan för textklassificering.

Huvudsteg:
- Läsa in det rensade utvecklingsurvalet
- Förbehandla textdata
- TF-IDF-vektorisering
- Skapa n-gram-features
- Förbereda feature-matrisen för modellering

In [1]:
import pandas as pd

reviews = pd.read_csv(
    "../data/processed/all_beauty_sample_100k.csv",
    keep_default_na=False
)

reviews.head()

,rating,title,text,asin,parent_asin,sentiment
0,5,Best lip balm ever,"love this lip balm, it’s very minting and crea...",B07HRG318F,B07HRG318F,positive
1,5,Convenient and comforting,I was so happy to have these on hand after my ...,B00E4MQENM,B00E4MQENM,positive
2,5,it works,"Great product, worth the money",B0758J3QV4,B079M24W38,positive
3,5,I will continue to buy from them,Great products and great customer service,B075WGGGTX,B0BGQQKWWY,positive
4,4,Breaks off,Didnt like that It eventually breaks off after...,B08LGHPNY3,B08LGHPNY3,positive


## Kontroll av data

Vi kontrollerar datasetets storlek, kolumner och fördelningen mellan sentimentklasserna innan textförbehandlingen.

In [2]:
print("Antal recensioner:", len(reviews))
print("\nKolumner:")
print(reviews.columns.tolist())

print("\nSentimentfördelning:")
print(reviews["sentiment"].value_counts())

Antal recensioner: 100000

Kolumner:
['rating', 'title', 'text', 'asin', 'parent_asin', 'sentiment']

Sentimentfördelning:
sentiment
positive    71303
negative    20667
neutral      8030
Name: count, dtype: int64


## Textförbehandling

Recensionstexten förbehandlas för att skapa en enhetlig textrepresentation inför TF-IDF och modellering. Texten konverteras till gemener och onödiga tecken tas bort.

In [3]:
import re

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

reviews["clean_text"] = reviews["text"].apply(preprocess_text)

reviews[["text", "clean_text"]].head()

,text,clean_text
0,"love this lip balm, it’s very minting and crea...",love this lip balm it s very minting and creat...
1,I was so happy to have these on hand after my ...,i was so happy to have these on hand after my ...
2,"Great product, worth the money",great product worth the money
3,Great products and great customer service,great products and great customer service
4,Didnt like that It eventually breaks off after...,didnt like that it eventually breaks off after...


### Tokenisering

Den rensade texten delas upp i enskilda ord (tokens). Detta gör det möjligt att inspektera hur texten representeras innan TF-IDF och n-grams skapas.

In [4]:
reviews["tokens"] = reviews["clean_text"].str.split()

reviews[["clean_text", "tokens"]].head()

,clean_text,tokens
0,love this lip balm it s very minting and creat...,"[love, this, lip, balm, it, s, very, minting, ..."
1,i was so happy to have these on hand after my ...,"[i, was, so, happy, to, have, these, on, hand,..."
2,great product worth the money,"[great, product, worth, the, money]"
3,great products and great customer service,"[great, products, and, great, customer, service]"
4,didnt like that it eventually breaks off after...,"[didnt, like, that, it, eventually, breaks, of..."


## TF-IDF och n-grams

Den förbehandlade texten omvandlas till numeriska features med TF-IDF. 
Både unigrams (enskilda ord) och bigrams (kombinationer av två ord) används för att fånga både enskilda ord och korta uttryck i recensionerna.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=10000,
    min_df=5,
    stop_words="english"
)

X = tfidf.fit_transform(reviews["clean_text"])
y = reviews["sentiment"]

print("Feature-matrisens dimension:", X.shape)
print("Antal TF-IDF-features:", len(tfidf.get_feature_names_out()))

Feature-matrisens dimension: (100000, 10000)
Antal TF-IDF-features: 10000


### Kontroll av skapade features

Nedan visas ett urval av de features som skapats med TF-IDF. Feature-matrisen innehåller totalt 10 000 unigrams och bigrams.

In [6]:
feature_names = tfidf.get_feature_names_out()

print("Exempel på features:")
print(feature_names[:50])

Exempel på features:
['aa' 'ab' 'ability' 'able' 'able fit' 'able make' 'able use' 'able wear'
 'abrasive' 'absolute' 'absolute best' 'absolute favorite' 'absolutely'
 'absolutely amazing' 'absolutely beautiful' 'absolutely gorgeous'
 'absolutely love' 'absolutely loved' 'absolutely loves'
 'absolutely perfect' 'absolutely recommend' 'absorb' 'absorb skin'
 'absorbed' 'absorbed skin' 'absorbent' 'absorbing' 'absorbs'
 'absorbs quickly' 'absorbs skin' 'absorption' 'ac' 'accent' 'accept'
 'acceptable' 'access' 'accessories' 'accessory' 'accident' 'accidentally'
 'accommodate' 'according' 'account' 'accurate' 'accurately' 'acetone'
 'aches' 'achieve' 'acid' 'acne']


## Feature-matris för modellering

TF-IDF-representationen används som feature-matris (`X`) och sentimentklassen används som målvariabel (`y`). `rating` används inte som input-feature eftersom sentimentvariabeln är härledd från rating och detta annars skulle orsaka target leakage.

Feature-matrisen är nu redo att användas i efterföljande klassificeringsmodeller.

In [7]:
print("X:", X.shape)
print("y:", y.shape)
print("\nMålklasser:", y.unique())

X: (100000, 10000)
y: (100000,)

Målklasser: <StringArray>
['positive', 'negative', 'neutral']
Length: 3, dtype: str


## Spara preprocessing-objekt

TF-IDF-vektoriseraren sparas så att samma textrepresentation kan återanvändas i nästa steg av projektet. Detta säkerställer att modellerna använder samma preprocessing och feature-definition.

In [8]:
import joblib
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(tfidf, output_dir / "tfidf_vectorizer.joblib")

print("TF-IDF-vektoriseraren har sparats.")

TF-IDF-vektoriseraren har sparats.


## Resultat

Textförbehandlingen och feature engineering är slutförd.

- 100 000 rensade recensioner används.
- Recensionstexten har normaliserats och tokeniserats.
- TF-IDF används för numerisk textrepresentation.
- Unigrams och bigrams används.
- Feature-matrisen innehåller 10 000 features.
- `X` innehåller TF-IDF-features och `y` innehåller sentimentklassen.
- `rating` används inte som feature för att undvika target leakage.

Feature-matrisen är redo för nästa steg: klassificering och modellutvärdering.